# 07 · Chronos-2 report figure from a notebook-05 run

Notebook 05 runs on the GX10 GPU and writes its results without a figure, so this notebook draws
the same report as notebooks 03 and 04 from a finished run folder: MAE by lead hour for the
fine-tuned and zero-shot models against the naive 168 h and 24 h baselines (test solid, replay
dashed), and one replay path with its 10-90% interval. The naive baselines are rebuilt from
`data/processed_hourly.parquet` with the definitions of notebook 03, after checking that its
labels are the ones the run scored. Runs locally on CPU; the figure lands in the run folder as
`report.png`, like 03's and 04's.

In [ ]:
# Run folder written by notebook 05, relative to the repo root (runs/gx10/nb05/... after gx10_pull)
RUN = "runs/gx10/nb05/c13-chronos2-h168"
TARGETS = ["grid_export", "grid_import"]
EXAMPLE_MODEL = "chronos2"                  # the replay path: "chronos2" | "chronos2_zeroshot"
EXAMPLE_DEVICE = None                      # None: the first device with the target

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
DATA_DIR = NB_DIR / "data"
RUN_DIR = REPO_ROOT / RUN
LOCAL_TZ = "Europe/Rome"

RES = pd.concat([pd.read_parquet(RUN_DIR / f"{block}.parquet").assign(block=block)
                 for block in ["test", "replay"]], ignore_index=True)
RES = RES[RES["target"].isin(TARGETS)]
PT = pd.read_parquet(DATA_DIR / "processed_hourly.parquet")
# One continuous hourly frame per device, so a lag is a plain index lookup.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}

# The naive baselines of notebook 03, per row. They are only valid if the data still holds the
# labels the run scored, so check that first.
parts = []
for (d, target), g in RES.groupby(["device_id", "target"]):
    s, t, h = DEV[d][target], pd.DatetimeIndex(g["ts_hour"]), g["horizon"].to_numpy()
    assert np.allclose(s.reindex(t).to_numpy(), g["actual"], equal_nan=True), \
        f"{d} {target}: processed_hourly.parquet no longer matches the run's labels"
    parts.append(g.assign(
        naive168=s.reindex(t - pd.Timedelta(hours=168)).to_numpy(),
        # yesterday's same hour, or the most recent one observable at the origin
        naive24=s.reindex(t - pd.to_timedelta(24 * np.ceil(h / 24), unit="h")).to_numpy()))
RES = pd.concat(parts, ignore_index=True)
print("run:", RUN, "| models:", sorted(RES["model"].unique()), "| devices:",
      RES["device_id"].nunique(), "| rows:", len(RES))

In [ ]:
# The scores notebook 05 computed (kWh/h), the same table as 03's and 04's.
display(pd.read_csv(RUN_DIR / "summary.csv", index_col=[0, 1, 2, 3]).round(4))

In [ ]:
COLORS = {"chronos2": "#3b6fb6", "chronos2_zeroshot": "#6a3d9a", "naive168": "#999999",
          "naive24": "#d9822b"}
# The naive baselines as models of their own, on the rows of one Chronos model (both share rows).
base = RES[RES["model"] == EXAMPLE_MODEL]
LONG = pd.concat([RES, *[base.assign(model=name, point=base[name])
                         for name in ["naive168", "naive24"]]], ignore_index=True)
fig, axes = plt.subplots(len(TARGETS), 2, figsize=(16, 4.5 * len(TARGETS)), squeeze=False)
for row, target in zip(axes, TARGETS, strict=True):
    one = LONG[LONG["target"] == target]
    lead = one.assign(ae=(one["point"] - one["actual"]).abs()).groupby(
        ["block", "model", "horizon"])["ae"].mean()
    for (block, model), s in lead.groupby(level=["block", "model"]):
        s.droplevel([0, 1]).plot(ax=row[0], color=COLORS[model], lw=2 if block == "test" else 1,
                                 ls="-" if block == "test" else "--", label=f"{model} {block}")
    row[0].set(title=f"{target}: MAE by lead hour", xlabel="lead (h)", ylabel="kWh/h")
    row[0].legend()
    path = one[(one["block"] == "replay") & (one["model"] == EXAMPLE_MODEL)]
    path = path[path["device_id"] == (EXAMPLE_DEVICE or path["device_id"].iloc[0])]
    path = path[path["origin"] == path["origin"].min()]
    row[1].fill_between(path["ts_hour"], path["q0.1"], path["q0.9"], color=COLORS[EXAMPLE_MODEL],
                        alpha=0.2, label="10-90%")
    row[1].plot(path["ts_hour"], path["actual"], color="#222222", lw=2, label="actual")
    row[1].plot(path["ts_hour"], path["point"], color=COLORS[EXAMPLE_MODEL], lw=2,
                label=EXAMPLE_MODEL)
    start = path["origin"].iloc[0].tz_convert(LOCAL_TZ)
    row[1].set(title=f"{target}: replay, {path['device_id'].iloc[0]} from {start:%Y-%m-%d %H:%M}"
               " local", ylabel="kWh/h")
    row[1].tick_params(axis="x", rotation=30)
    row[1].legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "report.png", dpi=110)